<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L05-rubrics-as-rewards/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L05-rubrics-as-rewards/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L05-rubrics-as-rewards/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L05-rubrics-as-rewards/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L05 · Rubrics as rewards: scoring open-ended answers with weighted checks, and how a policy learns to game them

**You will build:** the reward half of reinforcement learning on open-ended answers: a weighted
checklist reward, two audit statistics (false credit and agreement with the truth), a penalty on
padding, and an early-stopping monitor. You will train a tiny policy against the checklist and
watch what it learns to do to the checklist.

**Time:** ~60 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T04-L01-grpo-from-scratch (the group-relative policy-gradient update this
lesson reuses on a much smaller policy, so that the reward is the subject).

Maths and code have answers a program can check. A health-information answer does not: there is
no single right string. The setting here is an assistant for a community health programme in
India or Kenya, and the usual response is a **rubric**: a list of things a good answer does, each
with points, graded by a model. This lesson contains no medical or local-health fact and is not
medical advice; the "answers" are choices on a few abstract slots, so that every number can be
computed exactly and the reward design is all there is to look at.

By the end you will be able to:

1. Implement a weighted-checklist reward, and use it to train a softmax policy with a
   group-relative policy-gradient update.
2. Implement the false-credit rate of a grader and the pairwise agreement of a reward with a
   hidden truth, and use them to find which criteria a trained policy satisfies without doing
   the work.
3. Measure rubric reward against hidden true quality over training steps and over seeds, and
   count the seeds in which true quality peaks and then falls.
4. Implement a penalty on named padding channels, and measure which channels the policy keeps
   exploiting when only some of them are penalised.
5. Implement early stopping on a held-out monitor, and compare a monitor that shares the
   training grader's blind spot with an independently written one by the true quality of the
   checkpoint each one keeps.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T04-L01`

**This lesson assumes you can already:**

- write Python functions over numpy arrays and select with boolean masks and sums along an axis
- say what a policy-gradient update does to a softmax over choices, and what a group-relative advantage is (T04-L01 builds both)
- read a mean over several random seeds and say what it hides

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Rubrics as Rewards: Reinforcement Learning Beyond Verifiable Domains](https://arxiv.org/abs/2507.17746) · Gunjal et al. (arXiv 2507.17746) · *free* — sections 2.1 and 2.2 define the weighted checklist reward this lesson implements and the implicit single-score alternative it does not, and its ablations show what rubric design does and does not move
- [HealthBench: Evaluating Large Language Models Towards Improved Human Health](https://arxiv.org/abs/2505.08775) · Arora et al. (arXiv 2505.08775) · *free* — section 2 and appendix D define per-criterion points and the score normalisation used here; section 6.6 is its own check for length bias
- [Scaling Laws for Reward Model Overoptimization](https://arxiv.org/abs/2210.10760) · Gao, Schulman and Hilton (arXiv 2210.10760) · *free* — the measured gold-versus-proxy curves that sections 5 to 8 reproduce in miniature, with a gold reward model standing in for humans
- [OpenIntro Statistics](https://www.openintro.org/book/os/) · Diez, Cetinkaya-Rundel and Barr · *free* — chapter 5, on point estimates and sampling variability, explains the spread that a mean over several random seeds hides, which every table of this lesson averages away
- [NumPy, the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · The NumPy project · *free* — arrays, boolean masks and sums along an axis, the few numpy operations the five exercises lean on

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import itertools
import os
import signal
import sys
import threading
import time
import traceback
from typing import Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261014          # this lesson's one fixed seed: every training run derives from it, so
                          # re-running prints the same numbers anywhere.
N_SLOTS = 6              # an answer is one choice on each of six slots ...
N_OPTIONS = 4            # ... and every slot offers the same number of options
N_SEEDS = 16             # training runs per condition
N_STEPS = 400            # policy updates per run
GROUP = 32               # answers sampled per update (one "prompt", one group)
LEARNING_RATE = 0.15
EVAL_EVERY = 5           # a run is measured every EVAL_EVERY updates

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("rubric_reward",),
    "exercise 2": ("false_credit_rate",),
    "exercise 3": ("pair_agreement",),
    "exercise 4": ("penalised_reward",),
    "exercise 5": ("early_stop_step",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (pair_agreement)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _read_only(array: np.ndarray) -> np.ndarray:
    """Switch off writing to an array that the notebook shares with your code, and return it. Your
    functions are handed these arrays themselves, not copies: a function that edits its input in
    place (`points /= total`, `curve *= 1000`) would change what every later cell prints, with no
    error. On a read-only array the same line stops at once with numpy's own message. Given."""
    array.setflags(write=False)
    return array


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check below runs under a time limit far above what a correct answer needs, and so does every
# call a demo makes to your code (one training run, one table of scores, one audit statistic);
# a whole demo gets a much longer limit, because slow but correct code is allowed to be slow.
_CHECK_SECONDS = 2
_CALL_SECONDS = 2
_DEMO_SECONDS = 90


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that whoever started us had already set
    keeps its own deadline if it is due first, and is put back, with the time it had left and
    its repeat interval, when the block ends. A correct answer finishes far inside the limit,
    and nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose "
               "stopping condition is never met (a counter that is never "
               "advanced, an index that wraps round, a tolerance that is never "
               "reached exactly) runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        edits_input = (" (a function of yours writes into an array it was given: build a new array "
                       "instead of editing the input in place)" if "read-only" in str(exc) else "")
        print(f"{label}: raised {type(exc).__name__}: {exc}{edits_input}")
    else:
        _STATUS[label] = "passed"

## 1. The setting: an answer, four checklists and a hidden truth

An **answer** is one option on each of six slots, four options per slot. Three slots are what the
answer *says*: `warnings` (from no warning signs up to specific signs plus what to do),
`facility` (from nothing up to naming the right kind of local facility) and `plain` (reading
ease). Three are how it is *dressed*: `length` (3, 6, 9 or 12 sentences), `hedging` (0 to 3
hedging phrases) and `format` (plain paragraphs up to headed, bulleted sections). The policy you
will train picks one option per slot. Four options on six slots make every possible answer
countable, so every expected score in this lesson is computed exactly, not estimated.

A **rubric** is a list of criteria with points; a negative point value marks behaviour you do not
want. HealthBench, a benchmark of health conversations graded this way, sums the points of the
criteria that were met and divides by the points available. Rubrics as Rewards (RaR), which trains
a model on such rubrics with GRPO, has an explicit aggregation that does the same with weights, and
an implicit one that hands the whole rubric to a judge for a single score (exercise 1 says more).
`claims.yaml` has the sources and the quotes, read on 14 October 2026, and what this lesson says
about those papers is as of that date. Four checklists appear below, and the point of the lesson
is who sees which:

* the **training rubric** is the reward. The policy is optimised against it, through a grader.
* the **same-grader slice** is three more criteria written by the same team and graded by the
  same grader, kept out of the reward.
* the **independent rubric** is a second team's rubric, written from what the truth checks, and
  graded strictly (it means exactly what each criterion says). It is also kept out of the reward.
  It is the best case for a monitor, and the cell below prints what it covers and what it does not.
* the **truth** is a hidden checklist standing in for what a panel of clinicians would approve. It
  has pitfalls, criteria with negative points for what an answer should not do. The policy never
  sees it and a real team never has it; you can compute it here only because you can read the
  code that makes this world.

Read the cell below; nothing in it is graded.

In [ ]:
SLOT_NAMES = ("warnings", "facility", "plain", "length", "hedging", "format")
SENTENCES = (3, 6, 9, 12)          # sentences in an answer, for each option of the `length` slot
ALL_ANSWERS = _read_only(np.array(list(itertools.product(range(N_OPTIONS), repeat=N_SLOTS))))   # every answer


def option(answers, slot: str) -> np.ndarray:
    """The option chosen on one slot, for every answer in a batch. Given."""
    return np.asarray(answers)[:, SLOT_NAMES.index(slot)]


def sentence_count(answers) -> np.ndarray:
    return np.array(SENTENCES)[option(answers, "length")]


def padding(answers) -> np.ndarray:
    """How thorough an answer LOOKS: the length, hedging and format options added up (0 to 9)."""
    return option(answers, "length") + option(answers, "hedging") + option(answers, "format")


def channel_amounts(answers) -> np.ndarray:
    """Per answer: its sentence count, its hedging phrases and its format level. Given."""
    return np.stack([sentence_count(answers), option(answers, "hedging"),
                     option(answers, "format")], axis=1).astype(float)


# --- the training rubric: the reward -------------------------------------------------------
TRAIN_NAMES = ("names a specific warning sign",
               "says what to do when a warning sign appears",
               "uses plain language",
               "says what the assistant cannot know",
               "answers the caregiver's likely follow-up questions",     # soft
               "covers the cost and travel of getting care",              # soft
               "explains why each warning sign matters")                  # soft
TRAIN_POINTS = _read_only(np.array([5, 4, 3, 2, 3, 3, 3], dtype=float))
SOFT_BARS = (3, 5, 7)     # padding at which the grader credits the three soft criteria (section 3)


def truly_met_train(answers) -> np.ndarray:
    """What is really true of each answer, criterion by criterion. The three soft criteria need
    facts the assistant does not have (opening hours, the cost of the bus), so no answer meets
    them. Given."""
    warn, plain, hedge = option(answers, "warnings"), option(answers, "plain"), option(answers, "hedging")
    never = np.zeros(len(warn), dtype=bool)
    return np.stack([warn >= 2, warn >= 3, plain >= 2, hedge >= 1, never, never, never], axis=1)


# --- the same-grader slice and the independent rubric: monitors, never rewards -------------
SAME_NAMES = ("tells the caregiver why early care matters",     # soft
              "points to a follow-up visit",                    # soft
              "uses everyday words throughout")
SAME_POINTS = _read_only(np.array([3, 3, 2], dtype=float))
SAME_BARS = (4, 6)


def judged_same(answers) -> np.ndarray:
    """The SAME grader as the training rubric, on the slice that is kept out of the reward."""
    return np.stack([padding(answers) >= SAME_BARS[0], padding(answers) >= SAME_BARS[1],
                     option(answers, "plain") >= 3], axis=1)


STRICT_NAMES = ("names a specific warning sign",
                "says what to do when a warning sign appears",
                "uses plain language",
                "short enough to read in one sitting (at most 6 sentences)",
                "warnings not buried under hedges (at most one)",
                "no more structure than needed (at most a short list)")
STRICT_POINTS = _read_only(np.array([4, 3, 2, 4, 3, 2], dtype=float))
STRUCTURE = 5     # which criterion of the independent rubric is the one about structure


def met_strict(answers) -> np.ndarray:
    """The independent rubric, graded exactly as written. Given."""
    warn = option(answers, "warnings")
    return np.stack([warn >= 2, warn >= 3, option(answers, "plain") >= 2,
                     option(answers, "length") <= 1, option(answers, "hedging") <= 1,
                     option(answers, "format") <= 1], axis=1)


# --- the truth: hidden from the policy ------------------------------------------------------
# HealthBench's way of writing undesirable behaviour: a criterion with NEGATIVE points, met when the
# answer commits it. The independent rubric above asks about the same three things the other way
# round ("short enough"), which is why you will see the same behaviour counted twice in two forms.
TRUTH_NAMES = ("names a specific warning sign",
               "says what to do when a warning sign appears",
               "names the right kind of local facility",
               "uses plain language",
               "says what the assistant cannot know",
               "runs on past what a caregiver will read (more than 6 sentences)",
               "buries the warning under hedges (two or more)",
               "adds more structure than a short message needs (headed sections or more)")
TRUTH_POINTS = _read_only(np.array([5, 4, 4, 3, 2, -5, -4, -3], dtype=float))


def met_truth(answers) -> np.ndarray:
    """The hidden checklist, graded exactly. A pitfall is met when the answer commits it. Given."""
    warn = option(answers, "warnings")
    return np.stack([warn >= 2, warn >= 3, option(answers, "facility") >= 2,
                     option(answers, "plain") >= 2, option(answers, "hedging") >= 1,
                     option(answers, "length") >= 2, option(answers, "hedging") >= 2,
                     option(answers, "format") >= 2], axis=1)


def point_total(verdicts, points) -> np.ndarray:
    """The points of the criteria met, before any division by the points available. Whole numbers
    held in floats, so sums and ties are exact whatever order they are added in. Given."""
    return np.asarray(verdicts, dtype=float) @ np.asarray(points, dtype=float)


def covered(verdicts, truth_verdicts) -> list:
    """Which criteria of the truth some criterion in `verdicts` decides in exactly the same way, or
    in exactly the opposite way (a pitfall written as its mirror image), on every answer. Given."""
    verdicts = np.asarray(verdicts, dtype=bool)
    truth_verdicts = np.asarray(truth_verdicts, dtype=bool)
    return [j for j in range(truth_verdicts.shape[1])
            if ((verdicts == truth_verdicts[:, [j]]).all(axis=0) | (verdicts != truth_verdicts[:, [j]]).all(axis=0)).any()]


def _show_rubrics() -> None:
    sets = (("training rubric (the reward; soft criteria are judged leniently, see section 3)",
             TRAIN_NAMES, TRAIN_POINTS),
            ("same-grader slice (kept out of the reward)", SAME_NAMES, SAME_POINTS),
            ("independent rubric (kept out of the reward, graded strictly)", STRICT_NAMES, STRICT_POINTS),
            ("the truth (hidden from the policy)", TRUTH_NAMES, TRUTH_POINTS))
    print(f"{len(ALL_ANSWERS)} possible answers = {N_OPTIONS} options ** {N_SLOTS} slots")
    for title, names, points in sets:
        print(f"\n{title}   [{points[points > 0].sum():.0f} points available]")
        for name, pts in zip(names, points):
            print(f"  {pts:+4.0f}  {name}")
    absolute = np.abs(TRUTH_POINTS)
    print(f"\nwhat each checklist covers of the truth: a truth criterion counts as covered when some criterion of the"
          f" checklist decides every possible answer the same way, or the exact opposite way")
    for title, verdicts in (("training rubric (the checkable criteria)", truly_met_train(ALL_ANSWERS)),
                            ("same-grader slice (as graded)", judged_same(ALL_ANSWERS)),
                            ("independent rubric", met_strict(ALL_ANSWERS))):
        hit = covered(verdicts, met_truth(ALL_ANSWERS))
        missed = [TRUTH_NAMES[j] for j in range(len(TRUTH_NAMES)) if j not in hit]
        print(f"  {title:<42}covers {len(hit)} of {len(TRUTH_NAMES)} truth criteria, {absolute[hit].sum():.0f} of {absolute.sum():.0f} absolute points")
        print(f"    never asks about: " + ("; ".join(missed) if missed else "nothing"))


_show_rubrics()

## 2. Exercise 1 — `rubric_reward()`

A grader goes through the criteria one at a time and says met or not met. The reward turns those
verdicts into one number per answer. The convention here is HealthBench's, with one rule of this
lesson's own: **add up the points of the criteria that were met (a met criterion with negative
points subtracts), then divide by the points available, meaning the sum of the POSITIVE points
only.** An answer that meets every positive criterion and no negative one scores exactly 1. Three
details decide whether yours is right:

* Negative criteria go into the sum but never into the denominator.
* One answer's score can be negative, and it is not clipped. HealthBench clips the MEAN over
  examples to [0, 1] when it reports a score, not each answer, and a negative score is
  information a policy should feel. (`clipped_mean`, given below, is that reporting clip.)
* A rubric with no positive points has no maximum to divide by: every score is 0.0. This third
  detail is the lesson's own choice, not HealthBench's. Where its appendix meets a score with
  nothing positive to divide by (it does so for scores by axis, which use a subset of the
  criteria), it calls the score undefined and leaves that example out of the average. A training
  run cannot leave an answer out, so here the score is 0.0 and never a nan.

`met` may be a table of True and False or a table of 1 and 0; both are valid input. The arrays you
are given are the notebook's own, not copies, so build new arrays and never change an input in place.

RaR's explicit aggregation divides by the sum of all the weights and phrases its pitfalls
positively ("avoids ...") instead. Both are conventions for one idea; this lesson uses the
HealthBench one and says so wherever it matters. RaR also has an implicit aggregation, in which a
judge reads the whole rubric and gives one score, and in the paper's own comparison that variant did
best overall while the fixed weighted sums gave more control but could be brittle. This lesson builds
the weighted sum anyway, because it is the version whose every point you can audit. `claims.yaml`
has the sources.

<details><summary>💡 Hint 1 — what to think about</summary>

`met` is a table of yes/no verdicts, one row per answer, and `points` is one number per column.
What does one row's weighted sum look like as a single array operation? What is the number you
divide by, and does it depend on the answer? What would a pitfall that was met do to the sum,
and to the number you divide by?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn `met` into numbers and take its product with `points`, giving one weighted sum per answer.
Work out the points available once, from the positive entries of `points` only. Divide. Handle
the case where nothing positive is available before you divide, and make sure an input with no
rows returns an empty array rather than failing.
</details>

In [ ]:
def rubric_reward(met, points):
    """The checklist reward of each answer: the points of the criteria met, over the points available.

    `met[i, j]` is True (or 1) when the grader judged criterion j met in answer i, and `points[j]`
    is that criterion's points (negative for behaviour you do not want). The reward of answer i is
        (sum of points[j] over the criteria met in answer i) / (sum of the positive points).
    It is NOT clipped, so it can be negative. If no criterion has positive points, every reward is 0.0.
    `met` and `points` may be lists, tuples or numpy arrays, and the function must not modify them.

    Example:
        >>> rubric_reward([[True, True, False], [True, False, True], [False, False, True]], [5, 3, -4])
        array([ 1.   ,  0.125, -0.5  ])
        (answer 0 meets 5 and 3 of 8 available; answer 1 meets 5 and -4; answer 2 meets only -4)

    Returns:
        a float array with one reward per row of `met` (an empty array for no rows).
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement rubric_reward")


def clipped_mean(rewards) -> float:
    """What a report does with the per-answer rewards: their mean, clipped to [0, 1]. Given."""
    return float(np.clip(np.mean(rewards), 0.0, 1.0))


# Public checks — run these as often as you like.
def _check_rubric_reward() -> None:
    met = [[True, True, False], [True, False, True], [False, False, True]]
    got = rubric_reward(met, [5, 3, -4])
    assert isinstance(got, np.ndarray) and got.shape == (3,), (
        f"expected an array with one reward per answer, shape (3,); got {type(got).__name__} {getattr(got, 'shape', None)}"
    )
    assert np.allclose(got, [1.0, 0.125, -0.5], rtol=0, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected [1.0, 0.125, -0.5]: divide by the sum of the "
        "POSITIVE points only (8 here), and let a met pitfall subtract from the top"
    )
    assert got[2] < 0, (
        "answer 2 meets only a pitfall, so its reward must stay negative: the clip belongs to the "
        "reported mean (clipped_mean), not to each answer"
    )
    got = rubric_reward(np.array([[1, 1, 0], [1, 0, 1], [0, 0, 1]]), [5, 3, -4])
    assert np.allclose(got, [1.0, 0.125, -0.5], rtol=0, atol=1e-12), (
        f"a table of 1s and 0s gave {np.round(got, 4).tolist()}, expected [1.0, 0.125, -0.5]: it is valid input too. "
        "Multiply it by the points, but do not use it as an index into the points or flip it with ~, which "
        "read 1 and 0 as positions and as -2 and -1"
    )
    assert np.allclose(rubric_reward([[True, True]], [3, 1]), [1.0], rtol=0, atol=1e-12), (
        "meeting every positive criterion and no negative one must score exactly 1"
    )
    assert np.allclose(rubric_reward([[True, False], [False, True]], [-2, -1]), [0.0, 0.0], rtol=0, atol=1e-12), (
        "with no positive points there is nothing to divide by: every reward is 0.0, not nan and not an error"
    )
    assert rubric_reward(np.zeros((0, 3), dtype=bool), [1, 2, 3]).shape == (0,), (
        "no answers should give an empty array of rewards"
    )
    met_array, points_array = np.array([[1.0, 1.0, 0.0], [1.0, 0.0, 1.0]]), np.array([5.0, 3.0, -4.0])
    met_before, points_before = met_array.copy(), points_array.copy()
    rubric_reward(met_array, points_array)
    assert np.array_equal(met_array, met_before) and np.array_equal(points_array, points_before), (
        "your function changed an array it was given (dividing `points` by the points available, in place, is the usual "
        "way): a numpy array is passed by reference, so every later cell would be computed from the changed one. Make "
        "a new array instead"
    )
    print("exercise 1 looks right")


_try("exercise 1", _check_rubric_reward)

## 3. The grader, and where it leaks

Nobody reads a thousand answers per training step, so a model does the grading. A grader is an
imperfect reader, and this lesson plants one specific imperfection. Four of the seven training
criteria are checkable from the content of the answer, and the grader checks them correctly. The
last three ask for facts the assistant does not have (the clinic's opening hours, the price of the
bus), so no answer in this toy can truly meet them. A grader that cannot check a criterion does
what graders are known to do: it gives the benefit of the doubt to an answer that *looks*
thorough. Here, a soft criterion is credited once `padding` (the length, hedging and format
options added up) reaches a bar that differs from one soft criterion to the next.

That rule is a microscope, not a model of any real grader. Length and verbosity bias in model-based
graders and reward models is documented (Singhal et al. and Zheng et al.), and HealthBench checked
its own grader for it and found only a small correlation (`claims.yaml` quotes all three); the
question here is not whether yours does this, but what you would see if it did, and what would
catch it. Checklist rewards in practice also pair a judge with programs, though not by splitting
the criteria as this grader does: Viswanathan et al. put an AI judge on every checklist item and
average its score with a verifier program's wherever an item can be checked exactly (also in
`claims.yaml`).

The cell below defines the grader and scores four hand-made answers, against the training rubric
and against the truth. The truth has pitfalls, so the same four answers show what your exercise 1
does with a negative total. Read what it prints.

In [ ]:
def judged_train(answers) -> np.ndarray:
    """The grader's verdicts on the training rubric: the true ones, plus the benefit of the doubt
    on each soft criterion once the answer looks thorough enough. Given."""
    verdict = truly_met_train(answers)
    for column, bar in zip((4, 5, 6), SOFT_BARS):
        verdict[:, column] |= padding(answers) >= bar
    return verdict


FOUR_ANSWERS = {                  # options for: warnings, facility, plain, length, hedging, format
    "complete and concise": [3, 2, 3, 0, 1, 0],
    "complete but padded": [3, 2, 3, 3, 3, 3],
    "padded, little content": [1, 0, 1, 3, 3, 3],
    "bare": [0, 0, 0, 0, 0, 0],
}


def _four_answers() -> None:
    names = list(FOUR_ANSWERS)
    batch = np.array([FOUR_ANSWERS[n] for n in names])
    with _time_limit(_CALL_SECONDS, "rubric_reward in the four-answers demo"):
        reward = rubric_reward(judged_train(batch), TRAIN_POINTS)
        truth = rubric_reward(met_truth(batch), TRUTH_POINTS)
    print(f"{'answer':<24}{'sentences':>10}{'padding':>9}{'training reward':>17}{'true quality':>14}")
    for i, name in enumerate(names):
        print(f"{name:<24}{sentence_count(batch)[i]:>10}{padding(batch)[i]:>9}{reward[i]:>17.3f}{truth[i]:>14.3f}")
    by_reward = [names[i] for i in np.argsort(-reward, kind="stable")]
    by_truth = [names[i] for i in np.argsort(-truth, kind="stable")]
    print("\nbest first, by the training reward:", " > ".join(by_reward))
    print("best first, by the truth:          ", " > ".join(by_truth))
    print("the reward and the truth put the best answer " + ("in the same place." if by_reward[0] == by_truth[0]
                                                         else "in different places."))
    below = [n for n, t in zip(names, truth) if t < 0]
    print(f"\nmean true quality of the four: {truth.mean():.3f}; reported as HealthBench reports a score, clipped to [0, 1] "
          f"after the mean (clipped_mean): {clipped_mean(truth):.3f}")
    print(f"clip each answer at 0 first, the wrong way round, and the mean becomes {np.maximum(truth, 0).mean():.3f}"
          + (f": it would forgive {', '.join(repr(n) for n in below)} for scoring below zero." if below else "."))


_try("four answers", _four_answers, needs=("exercise 1",))

## 4. Exercise 2 — `false_credit_rate()`

How do you find a leak? Ask, criterion by criterion: of the answers that did **not** truly meet
it, how many did the grader credit anyway? That is the **false-credit rate**. A criterion that
every answer truly meets cannot be credited falsely, so its rate is `0.0` by convention (not
nan). The other error, a criterion the grader *missed* when it was truly met, is real but is a
different one and does not count here.

It needs the truth. Here you have it; a real team gets it from a small audited sample, where
people re-grade some answers and the verdicts are compared.

<details><summary>💡 Hint 1 — what to think about</summary>

Two tables of yes/no with the same shape: what the grader said and what was true. For each
column, which rows form the population the rate is about? Is that population the same size in
every column, and can it be empty?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Per column, count the rows where the criterion is truly unmet; that is the denominator. Count
among those the rows the grader marked met; that is the numerator. Divide column by column, and
choose what to return where the denominator is zero before you divide.
</details>

In [ ]:
def false_credit_rate(judged, truly):
    """For each criterion: of the answers that truly did NOT meet it, the share the grader credited.

    `judged[i, j]` is the grader's verdict (True or 1 = met) and `truly[i, j]` is what is really
    true (True or 1 = met), for answer i and criterion j. The rate of criterion j is
        (answers with judged True and truly False) / (answers with truly False),
    and 0.0 when no answer has truly False for that criterion. A criterion the grader MISSED
    (truly True, judged False) is a different error and does not count here.

    Example:
        >>> judged = [[True, False], [True, False], [True, True], [False, False]]
        >>> truly = [[False, False], [True, False], [False, False], [False, True]]
        >>> false_credit_rate(judged, truly)
        array([0.66666667, 0.33333333])
        (criterion 0 is truly unmet in answers 0, 2 and 3 and credited in 0 and 2; criterion 1 is
        truly unmet in answers 0, 1 and 2 and credited in answer 2 only)

    Returns:
        a float array with one rate per criterion.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement false_credit_rate")


# Public checks — run these as often as you like.
def _check_false_credit_rate() -> None:
    judged = [[True, False], [True, False], [True, True], [False, False]]
    truly = [[False, False], [True, False], [False, False], [False, True]]
    got = false_credit_rate(judged, truly)
    assert np.allclose(got, [2 / 3, 1 / 3], rtol=0, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected [0.6667, 0.3333]: the denominator is the number "
        "of answers that truly did NOT meet the criterion, counted per criterion, not the number of answers"
    )
    got = false_credit_rate([[True], [False]], [[True], [True]])
    assert np.allclose(got, [0.0], rtol=0, atol=1e-12), (
        "every answer truly meets this criterion, so nothing can be credited falsely: the rate is "
        "0.0, not nan and not an error"
    )
    got = false_credit_rate([[False, True]], [[True, False]])
    assert np.allclose(got, [0.0, 1.0], rtol=0, atol=1e-12), (
        "a criterion the grader missed (truly met, judged unmet) is not false credit; a criterion "
        "credited though truly unmet is, whatever the other criteria did"
    )
    got = false_credit_rate(np.array([[1, 0], [1, 0], [1, 1], [0, 0]]), np.array([[0, 0], [1, 0], [0, 0], [0, 1]]))
    assert np.allclose(got, [2 / 3, 1 / 3], rtol=0, atol=1e-12), (
        f"tables of 1s and 0s gave {np.round(got, 4).tolist()}, expected [0.6667, 0.3333]: they are valid input too. "
        "Convert them to True and False first (astype(bool)) before you invert one with ~, which turns 1 and 0 "
        "into -2 and -1"
    )
    assert false_credit_rate(np.zeros((0, 3), dtype=bool), np.zeros((0, 3), dtype=bool)).shape == (3,), (
        "no answers should still give one rate per criterion"
    )
    judged_array = np.array([[True, False], [True, False], [True, True], [False, False]])
    truly_array = np.array([[False, False], [True, False], [False, False], [False, True]])
    judged_before, truly_before = judged_array.copy(), truly_array.copy()
    false_credit_rate(judged_array, truly_array)
    assert np.array_equal(judged_array, judged_before) and np.array_equal(truly_array, truly_before), (
        "your function changed an array it was given (inverting the truth table in place is the usual way): a numpy "
        "array is passed by reference, so every later cell would be computed from the changed one. Make a new array instead"
    )
    print("exercise 2 looks right")


_try("exercise 2", _check_false_credit_rate)

In [ ]:
def _false_credit_at_the_start() -> None:
    truly = _read_only(truly_met_train(ALL_ANSWERS))
    with _time_limit(_CALL_SECONDS, "false_credit_rate in the false-credit demo"):
        rate = false_credit_rate(judged_train(ALL_ANSWERS), truly)
    print("a policy that picks every option with equal probability, so all possible answers count once")
    print(f"  {'criterion':<54}{'truly met':>10}{'false credit':>14}")
    for name, met, r in zip(TRAIN_NAMES, truly.mean(axis=0), rate):
        print(f"  {name:<54}{met:>10.3f}{r:>14.3f}")
    print(f"\nlargest false-credit rate among the checkable criteria: {rate[:4].max():.3f}; "
          f"among the soft ones: from {rate[4:].min():.3f} to {rate[4:].max():.3f}, before any training.")


_try("false credit at the start", _false_credit_at_the_start, needs=("exercise 2",))

## 5. Training a policy against the rubric

The policy is a softmax over the options of each slot, with the six slots chosen independently
and every option equally likely at the start. One update is the one you met in T04-L01: sample a
group of answers, score each with the reward, turn the rewards into **group-relative advantages**
(subtract the group's mean reward, divide by its standard deviation, as the GRPO of DeepSeekMath does;
`claims.yaml` quotes it), and move every slot's logits
along the advantage-weighted gradient of the log-probability of what was sampled. With one update
per batch the importance ratio is exactly 1, so the clip of the surrogate never acts, and there
is no KL term: this is the on-policy GRPO step on the smallest policy that can show the effect.
Everything in this section is given.

**Measuring.** With only a few thousand possible answers, the cell after the update code works
out the policy's *exact* expected score under each of the four checklists after every few
updates, by adding up probability times score over every possible answer. The curves therefore
have no sampling noise of their own. What varies from seed to seed is the training itself.

Read the cell below; nothing in it is graded.

In [ ]:
def option_probs(logits: np.ndarray) -> np.ndarray:
    """Softmax over the options of each slot: shape (slots, options). Given."""
    shifted = logits - logits.max(axis=1, keepdims=True)
    weights = np.exp(shifted)
    return weights / weights.sum(axis=1, keepdims=True)


def answer_probs(logits: np.ndarray) -> np.ndarray:
    """The probability the policy gives to each of the ALL_ANSWERS. Given."""
    return np.prod(option_probs(logits)[np.arange(N_SLOTS), ALL_ANSWERS], axis=1)


def sample_answers(logits: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    """Draw `n` answers from the policy, one uniform number per slot, by inverting the cumulative
    probabilities. Given."""
    cumulative = np.cumsum(option_probs(logits), axis=1)
    draws = rng.random((n, N_SLOTS))
    chosen = (draws[:, :, None] >= cumulative[None, :, :]).sum(axis=2)
    return np.minimum(chosen, N_OPTIONS - 1)


def group_advantages(rewards, eps: float = 1e-8) -> np.ndarray:
    """(reward - group mean) / (group standard deviation + eps), as in T04-L01. A group in which
    every answer scored the same gives advantages of (almost) zero and teaches nothing. Given."""
    rewards = np.asarray(rewards, dtype=float)
    return (rewards - rewards.mean()) / (rewards.std() + eps)


def policy_gradient(logits: np.ndarray, answers: np.ndarray, advantages: np.ndarray) -> np.ndarray:
    """Mean over the group of advantage * d log p(answer) / d logits: for each slot,
    (one-hot of the chosen option - the option probabilities). Given."""
    onehot = (answers[:, :, None] == np.arange(N_OPTIONS)).astype(float)
    return (advantages[:, None, None] * (onehot - option_probs(logits)[None])).mean(axis=0)


def train_policy(reward_fn: Callable[[np.ndarray], np.ndarray], seed: int) -> np.ndarray:
    """Train from the uniform policy for N_STEPS updates; return the logits every EVAL_EVERY
    updates (step 0 included): shape (N_STEPS // EVAL_EVERY + 1, slots, options). Given."""
    rng = np.random.default_rng([SEED, seed])
    logits = np.zeros((N_SLOTS, N_OPTIONS))
    saved = [logits.copy()]
    for step in range(1, N_STEPS + 1):
        answers = sample_answers(logits, GROUP, rng)
        advantages = group_advantages(reward_fn(answers))
        logits = logits + LEARNING_RATE * policy_gradient(logits, answers, advantages)
        if step % EVAL_EVERY == 0:
            saved.append(logits.copy())
    return np.array(saved)


# The score of every possible answer under each checklist, computed with YOUR rubric_reward, and
# the training runs made with your code, are kept so that a condition is trained once however many
# cells look at it. They are thrown away when you re-define the exercise they were made with and
# re-run a cell that needs them.
_TABLES: dict[str, np.ndarray] = {}
_TABLES_MADE_WITH: list = [None]
_RUNS: dict[str, tuple] = {}      # label -> (the exercise functions the runs were made with, the runs)


def tables() -> dict[str, np.ndarray]:
    if _TABLES_MADE_WITH[0] is not rubric_reward:
        _TABLES.clear()
        _TABLES_MADE_WITH[0] = rubric_reward
    if not _TABLES:
        with _time_limit(_CALL_SECONDS, "rubric_reward on all possible answers"):
            built = {"training": rubric_reward(judged_train(ALL_ANSWERS), TRAIN_POINTS),
                     "same": rubric_reward(judged_same(ALL_ANSWERS), SAME_POINTS),
                     "strict": rubric_reward(met_strict(ALL_ANSWERS), STRICT_POINTS),
                     "strict_narrow": rubric_reward(met_strict(ALL_ANSWERS)[:, :STRUCTURE], STRICT_POINTS[:STRUCTURE]),
                     "truth": rubric_reward(met_truth(ALL_ANSWERS), TRUTH_POINTS)}
        built.update({"sentences": sentence_count(ALL_ANSWERS).astype(float),
                      "hedges": option(ALL_ANSWERS, "hedging").astype(float),
                      "format": option(ALL_ANSWERS, "format").astype(float)})
        # a private, read-only copy of each table: what your function returned is not kept
        _TABLES.update({name: _read_only(np.array(table, dtype=float)) for name, table in built.items()})
    return _TABLES


def curves(history: np.ndarray) -> dict[str, np.ndarray]:
    """Exact expected scores of the policy at each saved step. Read-only, because the monitor demos
    hand these very arrays to your early_stop_step. Given."""
    probs = np.array([answer_probs(logits) for logits in history])
    return {name: _read_only(probs @ table) for name, table in tables().items()}


def run_condition(label: str, reward_fn: Callable[[np.ndarray], np.ndarray], uses_penalty: bool = False) -> list:
    """Train N_SEEDS policies against `reward_fn` and keep (history, curves) for each; kept by
    label, so a condition is trained once however many cells look at it, and made again if the
    exercise functions it uses have been re-defined. Each training run is time-limited, because
    it calls your reward code N_STEPS times. Given."""
    used = (rubric_reward, penalised_reward) if uses_penalty else (rubric_reward,)
    if label not in _RUNS or _RUNS[label][0] != used:
        runs = []
        for seed in range(N_SEEDS):
            with _time_limit(_CALL_SECONDS, f"one training run against '{label}' ({N_STEPS} updates, each calling your reward code)"):
                history = train_policy(reward_fn, seed)
            runs.append((history, curves(history)))
        _RUNS[label] = (used, runs)
    return _RUNS[label][1]


def training_reward(answers: np.ndarray) -> np.ndarray:
    """The reward the policy is trained on: the grader's verdicts, scored by YOUR rubric_reward."""
    return rubric_reward(judged_train(answers), TRAIN_POINTS)


def mean_curves(runs: list) -> dict[str, np.ndarray]:
    return {name: np.mean([c[name] for _, c in runs], axis=0) for name in runs[0][1]}


def _train_against_the_rubric() -> None:
    runs = run_condition("plain", training_reward)
    mean = mean_curves(runs)
    print(f"mean over {N_SEEDS} seeds; every number is an exact expectation under the policy at that step")
    print(f"{'step':>5}{'training':>10}{'same-grader':>13}{'independent':>13}{'TRUE':>8}{'sentences':>11}{'hedges':>8}{'format':>8}")
    for step in (0, 25, 50, 100, 200, N_STEPS):
        i = step // EVAL_EVERY
        print(f"{step:>5}{mean['training'][i]:>10.3f}{mean['same'][i]:>13.3f}{mean['strict'][i]:>13.3f}"
              f"{mean['truth'][i]:>8.3f}{mean['sentences'][i]:>11.1f}{mean['hedges'][i]:>8.2f}{mean['format'][i]:>8.2f}")
    truth = np.array([c["truth"] for _, c in runs])
    peak_milli = np.round(1000 * truth.max(axis=1)).astype(int)
    last_milli = np.round(1000 * truth[:, -1]).astype(int)
    fell = int(((peak_milli - last_milli) >= 10).sum())
    peak_step = EVAL_EVERY * truth.argmax(axis=1)
    print(f"\ntrue quality: start {truth[:, 0].mean():.3f}, mean peak {peak_milli.mean() / 1000:.3f} "
          f"(at step {peak_step.min()} to {peak_step.max()}), end {last_milli.mean() / 1000:.3f}")
    print(f"in {fell} of {N_SEEDS} seeds true quality ended at least 10 thousandths below its own peak; "
          f"the training reward's own best step was the last one in "
          f"{int(sum(c['training'].argmax() == len(c['training']) - 1 for _, c in runs))} of {N_SEEDS}.")


_try("train against the rubric", _train_against_the_rubric, needs=("exercise 1",))

### What the policy gave up

The training reward and the truth share some criteria, and the reward never mentions the rest. So
the policy had a reason to improve on the shared ones and none to keep the others where they were.
The next cell asks the hidden checklist, criterion by criterion, how likely an answer from the
policy is to meet it (for a pitfall, to commit it): at the start, at the training step where the
mean true quality peaks, and at the end. A criterion is flagged `worse` when the chance of meeting
it fell by more than a tenth between the start and the end, or, for a pitfall, when the chance of
committing it rose by more than a tenth, compared after rounding the change to three places.

In [ ]:
def _what_the_policy_gave_up() -> None:
    runs = run_condition("plain", training_reward)
    met = met_truth(ALL_ANSWERS).astype(float)
    peak_i = int(mean_curves(runs)["truth"].argmax())

    def meet_rate(i: int) -> np.ndarray:
        return np.mean([answer_probs(history[i]) @ met for history, _ in runs], axis=0)

    def worse(points: float, a: float, c: float) -> bool:
        return round(float(a - c if points > 0 else c - a), 3) > 0.1

    start, peak, end = meet_rate(0), meet_rate(peak_i), meet_rate(-1)
    print(f"chance that an answer meets each hidden criterion (mean over seeds); the peak column is step {peak_i * EVAL_EVERY}")
    print(f"  {'hidden criterion':<74}{'points':>7}{'start':>8}{'peak':>8}{'end':>8}")
    for name, pts, a, b, c in zip(TRUTH_NAMES, TRUTH_POINTS, start, peak, end):
        flag = "  worse" if worse(pts, a, c) else ""
        print(f"  {name:<74}{pts:>+7.0f}{a:>8.3f}{b:>8.3f}{c:>8.3f}{flag}")
    in_reward = [name in TRAIN_NAMES for name in TRUTH_NAMES]
    got_worse = [not inside and worse(p, a, c) for inside, p, a, c in zip(in_reward, TRUTH_POINTS, start, end)]
    print(f"\n{sum(got_worse)} of the {len(TRUTH_NAMES) - sum(in_reward)} hidden criteria that the reward never mentions got worse by more than a tenth.")


_try("what the policy gave up", _what_the_policy_gave_up, needs=("exercise 1",))

In [ ]:
def _false_credit_after_training() -> None:
    runs = run_condition("plain", training_reward)
    rates = []
    with _time_limit(_CALL_SECONDS, "false_credit_rate in the after-training demo"):
        for seed, (history, _) in enumerate(runs):
            answers = sample_answers(history[-1], 400, np.random.default_rng([SEED, 1000 + seed]))
            rates.append(false_credit_rate(judged_train(answers), truly_met_train(answers)))
        start = false_credit_rate(judged_train(ALL_ANSWERS), truly_met_train(ALL_ANSWERS))
    after = np.mean(rates, axis=0)
    print("false-credit rate of the soft criteria: the uniform start against 400 answers per seed from the trained policy")
    print(f"  {'criterion':<54}{'start':>8}{'trained':>9}")
    for name, a, b in zip(TRAIN_NAMES[4:], start[4:], after[4:]):
        print(f"  {name:<54}{a:>8.3f}{b:>9.3f}")


_try("false credit after training", _false_credit_after_training, needs=("exercise 1", "exercise 2"))

### Reading the three tables

Look at the training column and the TRUE column together, then at the sentence, hedge and format
columns, then at which hidden criteria got worse. The policy was never told to write long, hedged,
heavily formatted answers. It was told that the reward is higher when it does, because the grader
gives soft criteria to answers that look thorough, and the rubric was silent about every cost of
looking thorough. Padding is what the optimiser found; it is not something anyone asked for. Gao
et al. measured this shape, a proxy that rises while the gold reward peaks and falls, against a
fixed gold reward model (`claims.yaml`); here the gold standard is a checklist you can read.

Two things the tables do not say. They do not say the leak is large in a real system: you planted
it here, and HealthBench measured a small length effect on its own grader. They do not say the
peak of true quality is a good stopping point: you can only see the peak because you have the
truth. The next three exercises are about what you can do when you do not.

## 6. Exercise 3 — `pair_agreement()`

The tables compare curves. A sharper audit question is local: **given two answers, does the reward
order them the way the truth does?** That is exactly what a policy-gradient update uses, because it
only ever sees differences in reward between the answers of a group.

`pair_agreement(proxy, truth)` looks at every unordered pair of answers whose truth values are not
exactly equal. It scores the reward 1 for a pair it orders the way the truth does, 0 for a pair it
orders the other way, and one half for a pair the reward ties. The result is the average over
those pairs. Four details:

* A pair the TRUTH ties is skipped: there is no right order to agree with.
* A tie in the reward scores one half. A reward that cannot tell answers apart should land on
  0.5, which is what a coin would score; it is not agreement and it is not disagreement.
* "Tied" means exactly equal. Do not use a tolerance: values that are equal by construction, such
  as sums of whole-number points, are equal bit for bit, and two that differ in the last place
  are not tied.
* With no pair left to judge (fewer than two answers, or every truth value equal) the answer is
  0.5. Lists of different lengths are an error: raise `ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

For one pair, what are the three things the truth can say about its order, and the three things
the reward can say? Which of the nine combinations are skipped, which score 1, which 0 and which
one half? Then how do you list every pair exactly once?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For every pair `i < j`, compare the sign of the truth difference with the sign of the reward
difference. Drop the pairs where the truth sign is zero. Of the rest, count the pairs where the two
signs are equal as 1, the pairs where the reward sign is zero as one half, and divide by the number
of pairs kept. Decide what to return when none are kept before you divide.
</details>

In [ ]:
def pair_agreement(proxy, truth) -> float:
    """How often `proxy` orders two answers the way `truth` does.

    Every unordered pair (i < j) with truth[i] != truth[j] (exactly) counts once:
        1.0 if proxy orders the pair the same way as truth,
        0.0 if proxy orders it the opposite way,
        0.5 if proxy[i] == proxy[j] (exactly).
    The result is the mean over those pairs; 0.5 if there are none. Pairs where the truth is
    tied are skipped. Raise ValueError if the two lists differ in length.

    Example:
        >>> pair_agreement([0.2, 0.5, 0.9], [1.0, 2.0, 3.0])
        1.0
        >>> pair_agreement([0.5, 0.5, 0.9], [1.0, 2.0, 3.0])
        0.8333333333333334
        (pair 0-1 is tied in the reward and scores 0.5; pairs 0-2 and 1-2 agree; 2.5 / 3)

    Returns:
        a float in [0, 1].
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement pair_agreement")


# Public checks — run these as often as you like.
def _check_pair_agreement() -> None:
    got = pair_agreement([0.2, 0.5, 0.9], [1.0, 2.0, 3.0])
    assert got == 1.0, f"got {got!r}, expected 1.0: every pair is ordered the way the truth orders it"
    got = pair_agreement([0.9, 0.5, 0.2], [1.0, 2.0, 3.0])
    assert got == 0.0, f"got {got!r}, expected 0.0: every pair is ordered the opposite way"
    got = pair_agreement([0.5, 0.5, 0.9], [1.0, 2.0, 3.0])
    assert abs(got - 2.5 / 3) < 1e-12, (
        f"got {got!r}, expected 0.8333: a pair the reward ties scores one half, and the other two agree"
    )
    got = pair_agreement([1.0, 2.0, 3.0], [5.0, 5.0, 6.0])
    assert got == 1.0, (
        f"got {got!r}, expected 1.0: the pair the truth ties has no right order and is skipped, "
        "not counted as a disagreement or as an agreement"
    )
    got = pair_agreement([0.1 + 0.2, 0.3], [1.0, 2.0])
    assert got == 0.0, (
        f"got {got!r}, expected 0.0: 0.1 + 0.2 and 0.3 differ in the last place, so the reward orders "
        "this pair (the wrong way); 'tied' means exactly equal, with no tolerance"
    )
    assert pair_agreement([3.0, 1.0], [2.0, 2.0]) == 0.5 and pair_agreement([1.0], [1.0]) == 0.5, (
        "with no pair left to judge the answer is 0.5"
    )
    try:
        pair_agreement([1.0, 2.0], [1.0])
    except ValueError:
        pass
    else:
        raise AssertionError("lists of different lengths must raise ValueError")
    proxy_array, truth_array = np.array([0.4, 0.9, 0.1, 0.7]), np.array([0.3, 0.2, 0.8, 0.5])
    proxy_before, truth_before = proxy_array.copy(), truth_array.copy()
    pair_agreement(proxy_array, truth_array)
    assert np.array_equal(proxy_array, proxy_before) and np.array_equal(truth_array, truth_before), (
        "your function changed an array it was given (sorting or reordering them in place is the usual way): a numpy "
        "array is passed by reference, so every later cell would be computed from the changed one. Work on copies"
    )
    print("exercise 3 looks right")


_try("exercise 3", _check_pair_agreement)

The audit below feeds `pair_agreement` the **point totals** (`point_total`, given above), the
points of the criteria met before any division, instead of the divided rewards. Dividing every
score by the same positive number changes neither an order nor a tie, and whole-number totals tie
exactly however they are added up, so the table does not depend on how you wrote exercise 1.

In [ ]:
AGREE_STEPS = (0, 50, 100)      # training steps at which answers are drawn from the policy
N_AUDIT = 200                    # answers drawn per seed and step


def _tie_share(values) -> float:
    """The share of answer pairs a reward cannot tell apart (exactly equal values). Given."""
    values = np.asarray(values, dtype=float)
    first, second = np.triu_indices(len(values), k=1)
    return float((values[first] == values[second]).mean())


def _agreement_along_the_run() -> None:
    runs = run_condition("plain", training_reward)
    names = (("training rubric (the reward)", judged_train, TRAIN_POINTS),
             ("same-grader slice", judged_same, SAME_POINTS),
             ("independent rubric", met_strict, STRICT_POINTS))
    agree = {n: [] for n, _, _ in names}
    ties = []
    for step in AGREE_STEPS:
        per_seed = {n: [] for n, _, _ in names}
        tie_seed = []
        for seed, (history, _) in enumerate(runs):
            answers = sample_answers(history[step // EVAL_EVERY], N_AUDIT, np.random.default_rng([SEED, 2000 + seed]))
            truth = _read_only(point_total(met_truth(answers), TRUTH_POINTS))
            for n, judge, points in names:
                with _time_limit(_CALL_SECONDS, "pair_agreement in the agreement demo"):
                    per_seed[n].append(pair_agreement(point_total(judge(answers), points), truth))
            tie_seed.append(_tie_share(point_total(judged_train(answers), TRAIN_POINTS)))
        for n in per_seed:
            agree[n].append(float(np.mean(per_seed[n])))
        ties.append(float(np.mean(tie_seed)))
    print(f"agreement with the truth on {N_AUDIT} answers drawn from the policy at each step (mean over seeds; point totals)")
    print(f"  {'reward':<30}" + "".join(f"{'step ' + str(s):>10}" for s in AGREE_STEPS))
    for n, _, _ in names:
        print(f"  {n:<30}" + "".join(f"{v:>10.3f}" for v in agree[n]))
    print(f"  {'(0.5 is a coin)':<30}" + "".join(f"{0.5:>10.3f}" for _ in AGREE_STEPS))
    print(f"  {'pairs the training reward ties':<30}" + "".join(f"{v:>10.3f}" for v in ties))


_try("agreement along the run", _agreement_along_the_run, needs=("exercise 1", "exercise 3"))

### Reading the agreement table

Compare the rows. The same-grader slice and the training rubric were graded by the same grader
and share its blind spot; the independent rubric was written and graded differently. Which of the
three orders pairs more like the truth at the start, and which falls furthest as the policy
moves? The last row is the share of answer pairs the training reward cannot tell apart. A reward
that saturates ties more and more pairs, and a tie scores one half, so part of any fall towards
0.5 is the policy having used up what the rubric can tell apart.

## 7. Exercise 4 — `penalised_reward()`

The first repair to try is to charge for the behaviour. Length is the obvious channel, so
subtract a penalty that grows with it. A flat charge on every sentence would punish the thorough
answers that are good, so each channel gets a **free allowance**, and only the amount above it is
charged:

    penalty_i = sum over channels k of  slopes[k] * max(0, quantities[i, k] - allowances[k])
    reward_i  = base_i - penalty_i

`quantities[i, k]` is how much of channel `k` answer `i` uses (here: sentences, hedging phrases
and format level, from `channel_amounts()` above), `allowances[k]` is the free amount and
`slopes[k]` the cost per unit above it. The result is not clipped: the update only ever looks at differences
between the answers of a group, and clipping would flatten exactly the differences that matter.

The sweep after the exercise is the lesson: it penalises the channels you thought of first, then
all three, at several strengths, and measures what each choice does to the policy.

<details><summary>💡 Hint 1 — what to think about</summary>

Per answer and per channel there is an amount, an allowance and a slope. What is the amount ABOVE
the allowance when the answer is under it, and what should the charge be then? Which operation
turns a table of amounts into one number per answer?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Subtract the allowances from the amounts, replace anything below zero by zero, multiply each
column by its slope, and add up each row. Then subtract that from `base`. Do not change the
inputs, and do not clip the result.
</details>

In [ ]:
def penalised_reward(base, quantities, allowances, slopes):
    """`base` minus a charge for each channel's amount above its free allowance.

    `base[i]` is answer i's reward, `quantities[i, k]` its amount in channel k, `allowances[k]`
    the free amount and `slopes[k]` the cost per unit above it. For each answer
        penalty = sum over k of slopes[k] * max(0, quantities[i, k] - allowances[k])
    and the result is base - penalty. An amount at or below its allowance costs nothing and earns
    nothing, and an amount a fraction of a unit over costs that fraction of a slope. The result is
    not clipped. The inputs are not modified.

    Example:
        >>> penalised_reward([0.5, 0.5], [[6, 1, 1], [8, 2, 1]], [6, 1, 1], [0.25, 0.5, 0.5])
        array([ 0.5, -0.5])
        (answer 0 is within every allowance; answer 1 is 2 sentences and 1 hedge over: 2*0.25 + 1*0.5)

    Returns:
        a float array with one reward per answer.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement penalised_reward")


# Public checks — run these as often as you like.
def _check_penalised_reward() -> None:
    got = penalised_reward([0.5, 0.5], [[6, 1, 1], [8, 2, 1]], [6, 1, 1], [0.25, 0.5, 0.5])
    assert np.allclose(got, [0.5, -0.5], rtol=0, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected [0.5, -0.5]: charge slope * (amount - allowance) "
        "only above the allowance, per channel, and subtract the total from the base"
    )
    got = penalised_reward([0.25, 0.25, 0.25], [[5, 0, 0], [6, 1, 1], [7, 1, 1]], [6, 1, 1], [1.0, 1.0, 1.0])
    assert np.allclose(got, [0.25, 0.25, -0.75], rtol=0, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected [0.25, 0.25, -0.75]: below or at an allowance the "
        "charge is exactly zero (no bonus for being under it); one unit over costs one slope"
    )
    got = penalised_reward([0.5], [[12, 3, 3]], [6, 1, 1], [0.0, 0.5, 0.0])
    assert np.allclose(got, [-0.5], rtol=0, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected [-0.5]: a channel whose slope is 0 costs nothing "
        "however far over its allowance it is, and the result is not clipped at zero"
    )
    got = penalised_reward([0.5], [[6.5, 1.25, 1.0]], [6, 1, 1], [0.25, 0.5, 0.5])
    assert np.allclose(got, [0.25], rtol=0, atol=1e-12), (
        f"got {np.round(got, 4).tolist()}, expected [0.25]: the charge is the slope times the excess even when "
        "the excess is a fraction of a unit (6.5 sentences against 6 is half a unit over); do not round it "
        "up or down"
    )
    base_array, quantities_array = np.array([0.5, 0.5]), np.array([[6.0, 1.0, 1.0], [8.0, 2.0, 1.0]])
    base_before, quantities_before = base_array.copy(), quantities_array.copy()
    penalised_reward(base_array, quantities_array, [6, 1, 1], [0.25, 0.5, 0.5])
    assert np.array_equal(base_array, base_before) and np.array_equal(quantities_array, quantities_before), (
        "your function changed an array it was given (subtracting the charge from `base` in place is the usual way): "
        "return a new array instead"
    )
    print("exercise 4 looks right")


_try("exercise 4", _check_penalised_reward)

In [ ]:
PENALTY_UNITS = (0.005, 0.01, 0.02, 0.04)     # reward lost per sentence above the allowance
ALLOWANCES = _read_only(np.array([6.0, 1.0, 1.0]))        # free: 6 sentences, 1 hedging phrase, format level 1
CHANNEL_SLOPES = _read_only(np.array([1.0, 3.0, 3.0]))    # times the unit: a sentence, a hedge, a format level
                                               # (one option step is 3 sentences, 1 hedge, 1 level)
CHANNEL_SETS = {"LH": (1, 1, 0), "LHF": (1, 1, 1)}   # length and hedging; length, hedging and format


def penalised_training_reward(unit: float, channels: tuple) -> Callable[[np.ndarray], np.ndarray]:
    slopes = _read_only(unit * CHANNEL_SLOPES * np.array(channels, dtype=float))

    def reward_fn(answers: np.ndarray) -> np.ndarray:
        return penalised_reward(training_reward(answers), channel_amounts(answers), ALLOWANCES, slopes)
    return reward_fn


def _penalty_sweep() -> None:
    base = mean_curves(run_condition("plain", training_reward))
    unit_per = [f"{s:g}" for s in CHANNEL_SLOPES]
    print(f"end of training, mean over {N_SEEDS} seeds; the charge per `unit` is {unit_per[0]} per sentence above "
          f"{ALLOWANCES[0]:g}, {unit_per[1]} per hedging phrase above {ALLOWANCES[1]:g} and {unit_per[2]} per format "
          f"level above {ALLOWANCES[2]:g}; the rubric column is the training rubric's own reward, before any charge")
    print(f"  {'penalise':<10}{'unit':>7}{'rubric':>10}{'independent':>13}{'TRUE':>8}{'sentences':>11}{'hedges':>8}{'format':>8}")
    ends, along = {}, {}

    def row(label: str, unit: float, curve: dict) -> None:
        ends[label, unit] = {k: float(v[-1]) for k, v in curve.items()}
        along[label, unit] = curve["format"]
        print(f"  {label:<10}{unit:>7.3f}{curve['training'][-1]:>10.3f}{curve['strict'][-1]:>13.3f}"
              f"{curve['truth'][-1]:>8.3f}{curve['sentences'][-1]:>11.1f}{curve['hedges'][-1]:>8.2f}{curve['format'][-1]:>8.2f}")

    row("nothing", 0.0, base)
    for key, channels in CHANNEL_SETS.items():
        for unit in PENALTY_UNITS:
            row(key, unit, mean_curves(run_condition(f"penalty-{key}-{unit}", penalised_training_reward(unit, channels), True)))
    top = PENALTY_UNITS[-1]
    lh, lhf = ends["LH", top], ends["LHF", top]
    print("\nL = length, H = hedging, F = format.")
    print(f"at the strongest charge ({top:g}) with only L and H charged, the format level ended at {lh['format']:.2f} "
          f"(free up to {ALLOWANCES[2]:g}) and true quality at {lh['truth']:.3f}; charging all three: "
          f"format {lhf['format']:.2f}, true quality {lhf['truth']:.3f}, rubric reward {lhf['training']:.3f} "
          f"against {lh['training']:.3f}.")
    # The same strongest charge, followed along the run: where the third channel went, and when.
    steps = (50, 100, 200, N_STEPS)
    series = (("no charge", along["nothing", 0.0]), (f"L,H charged ({top:g})", along["LH", top]),
              (f"L,H,F charged ({top:g})", along["LHF", top]))
    hundredths = {label: [int(round(100 * values[s // EVAL_EVERY])) for s in steps] for label, values in series}
    print(f"\nthe format level along the run, mean over {N_SEEDS} seeds (free up to {ALLOWANCES[2]:g}; the most an answer can have is {N_OPTIONS - 1})")
    print(f"  {'step':<24}" + "".join(f"{s:>8}" for s in steps))
    for label, _ in series:
        print(f"  {label:<24}" + "".join(f"{h / 100:>8.2f}" for h in hundredths[label]))
    gaps = [a - b for a, b in zip(hundredths[series[1][0]], hundredths[series[0][0]])]
    shown = ", ".join(f"{g / 100:+.2f}" for g in gaps)
    if all(g > 0 for g in gaps):
        direction = "higher than with no charge at every step shown"
    elif all(g < 0 for g in gaps):
        direction = "lower than with no charge at every step shown"
    else:
        direction = "not consistently above or below the no-charge level"
    print(f"with only L and H charged the format level was {direction} ({shown}, compared at the 2 places printed); "
          f"with no charge at all it ended at {hundredths['no charge'][-1] / 100:.2f}.")


_try("penalty sweep", _penalty_sweep, needs=("exercise 1", "exercise 4"))

### Reading the penalty sweep

Read the L,H rows against the `nothing` row, then the L,H,F rows. Which columns did the charge
move, and which did it leave where they were? In the last table, look at when the format level
rose as well as where it ended. What the sweep supports is a limited statement: a charge on the
channels someone named leaves the exploit alive in the channel nobody named, and charging all three
is a different result from charging two. Whether a policy would find a fourth channel is not
something six softmaxes can show, and the last section says why that matters.

## 8. Exercise 5 — `early_stop_step()`

The second repair watches instead of charging. A **monitor** is a score you compute during
training and never optimise. When it stops improving you stop, and keep the best checkpoint it saw.
`early_stop_step(curve, patience, min_gain_milli)` reads one monitor's values, one per
evaluation, and returns the index of the checkpoint to keep:

* Values are compared as whole thousandths: `round(1000 * value)`, with Python's `round`.
* The best checkpoint so far starts as index 0. A later value is an **improvement** only if it
  beats the best checkpoint's value by at least `min_gain_milli` thousandths. Compare with the
  BEST so far, not with the previous value. A value that is not an improvement does not replace
  the best, and counts as one more evaluation without improvement, even when it is a little
  above the best or above the value just before it.
* After `patience` evaluations in a row without an improvement, stop looking. Return the index of
  the best checkpoint, not the index where you stopped. If the curve ends first, return its best.
* `patience` and `min_gain_milli` must be at least 1 and the curve must not be empty: raise
  `ValueError`.

<details><summary>💡 Hint 1 — what to think about</summary>

You need three pieces of state while you walk along the curve: which checkpoint is best so far,
its value in thousandths, and how many evaluations in a row have failed to beat it. What resets
the last one? What does a value that beats the best by less than the minimum gain do to each of
the three?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn every value into whole thousandths first. Start with index 0 as best and no failures. For
each later index, if its value is at least the minimum gain above the best's, it becomes the best
and the failure count resets; otherwise add one to the failure count and stop if it has reached
`patience`. Check the arguments before you start.
</details>

In [ ]:
def early_stop_step(curve, patience: int, min_gain_milli: int) -> int:
    """The index of the checkpoint to keep from a monitor curve.

    `curve` is a list, a tuple or a numpy array of floats. Values are compared as whole
    thousandths, `round(1000 * value)`. Index 0 is the best so far.
    Walk forward: a value that beats the best's by at least `min_gain_milli` thousandths becomes
    the new best and resets the count of failures; any other value (equal to the best, or only a
    little above it) adds one failure, and after `patience` failures in a row the walk stops. A
    failure does not replace the best and does not reset the count, even when it is higher than the
    value just before it. Return the index of the best, not of the stop.
    Raise ValueError if the curve is empty or if patience or min_gain_milli is below 1.

    Example:
        >>> early_stop_step([0.50, 0.60, 0.70, 0.69, 0.68, 0.80], patience=2, min_gain_milli=5)
        2
        (index 2 is the best; two evaluations in a row fail to beat it, so the 0.80 is never reached)

    Returns:
        an int, an index into `curve`.
    """
    # YOUR CODE HERE
    raise NotImplementedError("implement early_stop_step")


# Public checks — run these as often as you like.
def _check_early_stop_step() -> None:
    got = early_stop_step([0.50, 0.60, 0.70, 0.69, 0.68, 0.80], patience=2, min_gain_milli=5)
    assert got == 2, (
        f"got {got}, expected 2: return the index of the BEST checkpoint, not where the walk stopped "
        "(that would be 4), and stop once `patience` evaluations in a row have failed to beat it"
    )
    got = early_stop_step([0.50, 0.60, 0.70, 0.69, 0.80], patience=2, min_gain_milli=5)
    assert got == 4, (
        f"got {got}, expected 4: only one evaluation failed before 0.80 beat the best, so the count "
        "of failures resets and the walk goes on"
    )
    got = early_stop_step([0.500, 0.501, 0.502, 0.503, 0.504], patience=3, min_gain_milli=3)
    assert got == 3, (
        f"got {got}, expected 3: compare each value with the best so far, not with its neighbour: 0.503 is "
        "three thousandths above 0.500, 0.501 and 0.502 are not and do not replace the best"
    )
    assert early_stop_step([0.3, 0.302], patience=1, min_gain_milli=2) == 1, (
        "0.302 against 0.300 is a gain of 2 whole thousandths; subtracting floats gives 0.00199999..., "
        "so round to whole thousandths first and compare integers"
    )
    assert early_stop_step([0.4], patience=3, min_gain_milli=1) == 0, "a one-point curve keeps its only point"
    got = early_stop_step([0.5, 0.7, 0.701, 0.702, 0.703, 0.9], patience=3, min_gain_milli=5)
    assert got == 1, (
        f"got {got}, expected 1: 0.701, 0.702 and 0.703 are above the best (0.700) but by less than the minimum gain of "
        "5 thousandths, so each one is a failure. It neither replaces the best nor resets the count, and the third "
        "failure stops the walk before 0.9"
    )
    got = early_stop_step([0.50, 0.60, 0.60, 0.60, 0.70], patience=5, min_gain_milli=2)
    assert got == 4, (
        f"got {got}, expected 4: a value equal to the best is a failure like any other. Count it and move on "
        "to the next value; a branch that skips the count without moving on never ends"
    )
    got = early_stop_step([0.50, 0.60, 0.60, 0.60, 0.70], patience=2, min_gain_milli=1)
    assert got == 1, (
        f"got {got}, expected 1: two values in a row equal to the best are two failures, which is the patience, "
        "so the walk stops before it reaches 0.70; an equal value is not skipped, it counts"
    )
    for kind, curve in (("numpy array", np.array([0.1, 0.5, 0.4, 0.3])), ("tuple", (0.1, 0.5, 0.4, 0.3))):
        try:
            got = early_stop_step(curve, patience=2, min_gain_milli=1)
        except ValueError as exc:
            raise AssertionError(
                f"a {kind} curve raised ValueError ({exc}): the demos pass numpy arrays. Test `len(curve) == 0`, "
                "not `not curve`, which is ambiguous for an array"
            ) from None
        assert got == 1, f"a {kind} curve gave {got}, expected 1: it should work like a list"
    curve = np.array([0.1, 0.5, 0.4, 0.3])
    before = curve.copy()
    early_stop_step(curve, patience=2, min_gain_milli=1)
    assert np.array_equal(curve, before), (
        "your function changed the array it was given (scaling it to thousandths, in place, is the usual way): a numpy "
        "array is passed by reference and the notebook's own curves would be changed with it. Make a new list or array"
    )
    for bad in (([], 2, 1), ([0.1, 0.2], 0, 1), ([0.1, 0.2], 2, 0)):
        try:
            early_stop_step(*bad)
        except ValueError:
            continue
        raise AssertionError(f"arguments {bad} must raise ValueError")
    print("exercise 5 looks right")


_try("exercise 5", _check_early_stop_step)

In [ ]:
PATIENCE = 5          # evaluations in a row without a gain before the monitor gives up
MIN_GAIN_MILLI = 2    # a gain must be at least this many thousandths above the best so far


def _monitors() -> None:
    runs = run_condition("plain", training_reward)
    truth = np.array([c["truth"] for _, c in runs])
    last = truth.shape[1] - 1
    best = truth.max(axis=1)
    print(f"patience {PATIENCE} evaluations ({PATIENCE * EVAL_EVERY} updates), minimum gain {MIN_GAIN_MILLI} thousandths; "
          f"mean over {N_SEEDS} seeds")
    print(f"  {'monitor':<38}{'step kept':>10}{'TRUE quality there':>20}{'lost against hindsight':>24}")
    for label, key in (("the training rubric (the reward)", "training"), ("the same-grader slice", "same"),
                       ("the independent rubric", "strict"),
                       ("... without its structure criterion", "strict_narrow")):
        with _time_limit(_CALL_SECONDS, "early_stop_step in the monitors demo"):
            kept = np.array([early_stop_step(c[key], PATIENCE, MIN_GAIN_MILLI) for _, c in runs])
        quality = truth[np.arange(N_SEEDS), kept]
        print(f"  {label:<38}{EVAL_EVERY * kept.mean():>10.1f}{quality.mean():>20.3f}{(best - quality).mean():>24.3f}")
    print(f"  {'no monitor: the last step':<38}{EVAL_EVERY * last:>10.1f}{truth[:, last].mean():>20.3f}{(best - truth[:, last]).mean():>24.3f}")
    print(f"  {'hindsight: the best true step':<38}{EVAL_EVERY * truth.argmax(axis=1).mean():>10.1f}{best.mean():>20.3f}{0.0:>24.3f}")


_try("monitors", _monitors, needs=("exercise 1", "exercise 5"))

### A monitor says that something is wrong, and the criteria say where

Early stopping keeps a better checkpoint; it does not make the reward better. For that you need to
know *which* behaviour the policy is exploiting, and an independent rubric answers that too,
because it is a list of named criteria. The next cell reads it for two penalised runs: the one
that charged only length and hedging, and the one that charged all three channels.

In [ ]:
def _which_channel_leaks() -> None:
    top = PENALTY_UNITS[-1]
    met = met_strict(ALL_ANSWERS).astype(float)
    columns = {}
    for key in CHANNEL_SETS:
        runs = run_condition(f"penalty-{key}-{top}", penalised_training_reward(top, CHANNEL_SETS[key]), True)
        columns[key] = np.mean([answer_probs(history[-1]) @ met for history, _ in runs], axis=0)
    print(f"chance that an answer from the trained policy meets each independent criterion (charge {top:g})")
    print(f"  {'independent criterion':<62}{'L,H charged':>12}{'L,H,F charged':>15}")
    for i, name in enumerate(STRICT_NAMES):
        print(f"  {name:<62}{columns['LH'][i]:>12.3f}{columns['LHF'][i]:>15.3f}")
    worst = int(np.argmin(columns["LH"]))
    print(f"\nthe criterion the L,H run fails most: '{STRICT_NAMES[worst]}' ({columns['LH'][worst]:.3f}); "
          f"with all three channels charged it is {columns['LHF'][worst]:.3f}.")
    narrow = int(np.argmin(columns["LH"][:STRUCTURE]))
    print(f"a monitor written without the structure criterion ('{STRICT_NAMES[STRUCTURE]}') would show the L,H run's "
          f"weakest criterion as '{STRICT_NAMES[narrow]}' ({columns['LH'][narrow]:.3f}): what a monitor can name "
          f"is limited to what it asks about.")


_try("which channel leaks", _which_channel_leaks, needs=("exercise 1", "exercise 4"))

### How much of this depends on what the truth cares about?

Everything above used one hidden checklist. The size of the divergence is a statement about that
checklist: how much a reviewer minds an answer that is long, hedged and heavily formatted. The
cell below keeps the trained policies exactly as they are and changes only the points of the
truth's three pitfalls about dressing, then asks the same two questions again: how far does true
quality fall from its peak, and does the independent monitor still keep a good checkpoint? The
last row is a disclosure: those three weights were set after a first exploratory run, and the row
shows what the weights of that first draft give here, written as pitfalls like the others.

In [ ]:
DRESSING = (5, 6, 7)                       # the truth's pitfalls about length, hedging and structure
DRESSING_SCALES = (0.0, 0.25, 0.5, 1.0, 1.5)
FIRST_DRAFT = (4, 3, 2)                    # what those three behaviours were worth in the first exploratory run


def _depends_on_the_truth() -> None:
    runs = run_condition("plain", training_reward)
    met = _read_only(met_truth(ALL_ANSWERS))
    probs = np.array([[answer_probs(logits) for logits in history] for history, _ in runs])   # seeds x steps x answers
    with _time_limit(_CALL_SECONDS, "early_stop_step in the truth-dependence demo"):
        kept = np.array([early_stop_step(c["strict"], PATIENCE, MIN_GAIN_MILLI) for _, c in runs])
    rows = [(f"{scale:.2f}", scale * TRUTH_POINTS[list(DRESSING)]) for scale in DRESSING_SCALES]
    rows.append(("draft " + "/".join(str(p) for p in FIRST_DRAFT), -np.array(FIRST_DRAFT, dtype=float)))
    print(f"true quality of the same {N_SEEDS} trained policies when the points of the three dressing pitfalls are multiplied (mean over seeds)")
    print(f"  {'dressing x':>11}{'start':>8}{'peak':>8}{'end':>8}{'seeds that fell':>17}{'independent monitor':>21}")
    worse = []
    for label, dressing_points in rows:
        points = TRUTH_POINTS.copy()
        points[list(DRESSING)] = dressing_points
        _read_only(points)
        with _time_limit(_CALL_SECONDS, "rubric_reward in the truth-dependence demo"):
            scores = rubric_reward(met, points)
        truth = probs @ scores                                           # seeds x steps
        fell = int((np.round(1000 * truth.max(axis=1)) - np.round(1000 * truth[:, -1]) >= 10).sum())
        monitor, end = round(float(truth[np.arange(N_SEEDS), kept].mean()), 3), round(float(truth[:, -1].mean()), 3)
        if monitor < end:
            worse.append(label)
        print(f"  {label:>11}{truth[:, 0].mean():>8.3f}{truth.max(axis=1).mean():>8.3f}{truth[:, -1].mean():>8.3f}"
              f"{fell:>13} of {N_SEEDS}{monitor:>21.3f}")
    paid = ", ".join(f"{-p:g}" for p in TRUTH_POINTS[list(DRESSING)])
    first = ", ".join(str(p) for p in FIRST_DRAFT)
    available = TRUTH_POINTS[TRUTH_POINTS > 0].sum()
    draft_available = available + sum(FIRST_DRAFT)
    print("\n'seeds that fell' counts seeds whose true quality ended at least 10 thousandths under its own peak; the last")
    print(f"column is the true quality at the checkpoint the independent rubric keeps. The 1.00 row is the truth used above: its three")
    print(f"pitfalls cost {paid} points, raised from {first} after a first exploratory run in which the effect on true quality")
    print(f"looked small. The draft row writes those earlier weights as pitfalls too, over the same {available:g} points available, and the bar of")
    print(f"10 thousandths is on that {available:g}-point scale. The first exploratory run wrote the three behaviours as positive criteria over")
    print(f"{draft_available:g} points, so its own drops were {available / draft_available:.2f} times the size of the draft row's.")
    print("rows where the independent monitor kept a WORSE checkpoint than training to the end (compared at the 3 places printed): "
          + (", ".join(worse) if worse else "none"))


_try("depends on the truth", _depends_on_the_truth, needs=("exercise 1", "exercise 5"))

## 9. What this toy does not show, and common mistakes

**What it does not show.**

* That real graders leak in this way. The rule that credits soft criteria to thorough-looking
  answers is planted, to give you a leak with a known size. Length bias in model-based graders is
  documented, and HealthBench measured its own grader and found a small length correlation; both
  are in `claims.yaml`. Your grader's bias, if it has one, is an empirical question for an
  audited sample.
* That a hidden truth exists. Here it is a second checklist you can compute. In practice it is a
  panel of clinicians looking at a small sample, it is expensive and it is partial. The
  independent rubric is a best case: it restates most of what the truth checks, as the coverage
  printed in section 1 shows, and it is graded exactly. With a model grader neither property is
  free. The monitors table in section 8 shows what removing one of its criteria does, and the last
  table there varies how much the truth minds dressing.
* Anything about languages and places. Reward models behave differently across languages
  (M-RewardBench, `claims.yaml`), so a rubric written for one health system and graded in another
  language is a different rubric. The "local facility" criterion here is a placeholder, and the
  lesson holds no local fact.
* That the weights matter. One study of synthetic rubrics saw minimal differences from including
  weights or pitfall criteria in training (`claims.yaml`). The weights in this lesson are
  conventions chosen to make the world readable.
* What a language model can find. A policy that is six softmaxes can only move the six slots.
  A model can discover exploits nobody named, which is why a monitor has to be independent of the
  reward, not merely held out from it.

**Common mistakes.**

* **Dividing by the sum of the absolute points.** The points available are the positive ones.
* **Clipping each answer's score.** Clip the reported mean; a negative score is information.
* **Counting false credit over all answers.** The population is the answers that truly did not
  meet the criterion, criterion by criterion.
* **Giving a reward tie a zero or a one.** A tie is half. Ties are also what a saturated reward
  produces, so count them.
* **Treating a held-out slice of your own rubric as a monitor.** Same grader, same blind spot.
* **Charging for the exploit you thought of and declaring it fixed.** Check the other channels.
* **Returning the step where early stopping stopped.** Keep the best checkpoint, not the last
  one it looked at.

In [ ]:
def _bottom_line() -> None:
    plain_runs = run_condition("plain", training_reward)
    top = PENALTY_UNITS[-1]
    curves_of = {
        "start": {k: v[:1] for k, v in mean_curves(plain_runs).items()},
        "plain": {k: v[-1:] for k, v in mean_curves(plain_runs).items()},
        "LH": {k: v[-1:] for k, v in mean_curves(
            run_condition(f"penalty-LH-{top}", penalised_training_reward(top, CHANNEL_SETS["LH"]), True)).items()},
        "LHF": {k: v[-1:] for k, v in mean_curves(
            run_condition(f"penalty-LHF-{top}", penalised_training_reward(top, CHANNEL_SETS["LHF"]), True)).items()},
    }
    with _time_limit(_CALL_SECONDS, "early_stop_step in the bottom-line demo"):
        kept = [early_stop_step(c["strict"], PATIENCE, MIN_GAIN_MILLI) for _, c in plain_runs]
    curves_of["stopped"] = {k: np.array([np.mean([c[k][i] for (_, c), i in zip(plain_runs, kept)])])
                            for k in ("training", "strict", "truth")}
    rows = (("the uniform starting policy", "start"),
            ("trained on the rubric as written", "plain"),
            ("... keeping the checkpoint the independent rubric picks", "stopped"),
            ("... with length and hedging charged", "LH"),
            ("... with length, hedging and format charged", "LHF"))
    print(f"  {'the same task, five ways':<58}{'rubric':>10}{'independent':>13}{'TRUE':>8}")
    for label, key in rows:
        c = curves_of[key]
        print(f"  {label:<58}{c['training'][0]:>10.3f}{c['strict'][0]:>13.3f}{c['truth'][0]:>8.3f}")
    print("\nthe rubric column is the training rubric's own reward (before any charge, for the charged rows): the number a")
    print("dashboard of the reward would show. The TRUE column is the one you cannot see.")


_try("bottom line", _bottom_line, needs=("exercise 1", "exercise 4", "exercise 5"))

## 10. Self-check

1. During training the rubric reward climbs towards its maximum while an independent review of
   sampled answers gets worse. What is the most useful first step?
   - (a) Lower the learning rate until the two curves agree
   - (b) Add criteria to the rubric until the reward stops rising
   - (c) Take a small audited sample, compare the grader's verdicts with the audit criterion by
     criterion, and look for criteria that are credited without being met
   - (d) Report the reward curve, because the reviewers are the noisy measurement

2. A team keeps a fifth of its rubric's criteria out of the reward, graded by the same grader,
   and watches them during training. They rise along with the training reward. What follows?
   - (a) The policy is generalising, so the rubric is sound
   - (b) Nothing about quality: criteria that share the grader's blind spot rise with anything
     that fools it, so they cannot warn of that failure
   - (c) The held-out criteria are too easy and should be dropped
   - (d) The learning rate is too high

3. After a length penalty is added, answers stop growing, but an independent review still scores
   them poorly while the reward keeps rising. Which explanation fits best?
   - (a) Group-relative advantages cannot work with penalised rewards
   - (b) The penalty is too small by definition, so keep doubling it
   - (c) Length was never the problem; the policy is simply undertrained and needs more updates
   - (d) The exploit can keep paying through a channel the penalty does not name, such as
     formatting, so a penalty fixes only the behaviour someone thought of

4. A rubric's positive criteria total 10 points. An answer meets one positive criterion worth 2
   and one pitfall worth -6. Using this lesson's convention, what is the answer's score, and what
   does a report do with it?
   - (a) -0.4 for the answer; only the mean over answers is clipped to [0, 1]
   - (b) 0.0, because a score cannot be negative
   - (c) 0.2, because pitfalls are ignored when computing a reward
   - (d) -0.25, because the divisor is the sum of the absolute points

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")
    broken = [label for label, state in _STATUS.items() if label not in _EXERCISES and state == "failed"]
    if broken:
        print("a demo that runs your code failed: " + ", ".join(broken) + ". Its own cell above says why; "
              "a check that passes can still miss a case a demo reaches.")

## What you built, and where it goes next

`rubric_reward` turns a grader's verdicts into the number a policy-gradient trainer optimises.
`false_credit_rate` and `pair_agreement` are the two audit statistics for any grader you did not
write, computed on a small audited sample. `penalised_reward` and `early_stop_step` are the two
repairs, and you have measured what each does and does not cover.

Left out on purpose: a model as the grader, rubric generation, cross-language judging and
anything with real patient text. Each is worth its own lesson, and none can be studied honestly
until you can measure the simple case, which is what you just did.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_rubric_reward, ()),
        ("exercise 2", _check_false_credit_rate, ()),
        ("exercise 3", _check_pair_agreement, ()),
        ("exercise 4", _check_penalised_reward, ()),
        ("exercise 5", _check_early_stop_step, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.